# PC-FNO3D national — règle GFS v5.1, partie A (Colab CPU) : GFS f0 / f24 / f48

Règle `configs/national/regle_national_gfs_v5_1.json` (sha256 `570713841039…`), approuvée par Ines le 22/11/2026 (soir) et vérifiée par le carnet ; elle remplace `regle_national_gfs_v5` (jamais exécutée). Code : dépôt public au commit `b3ac84a93e34` (le carnet refuse de tourner tant qu'il n'est pas renseigné). Sorties : `MyDrive/pcfno3d_national/gfs_v5_regle/` et `runs/national_1km_v5_gfs` (dossiers neufs, rien n'est supprimé ni écrasé). Choix sur la validation 2021 seulement ; 2022 n'est pas relu ; l'échantillon 2026 (jours impairs, 00/06/12/18 UTC, ~608 instants) n'est lu qu'une fois, et seulement si une variante passe les trois cases sur 2021 puis si les parties vent de l'entrée de production sont identiques à celles du cache. GFS et ERA5 ne sont jamais appelés mesures (seuls les METAR le sont). Aucun bruit ajouté. Plafond de la règle : 90 unités Colab (A + B1 + B2) ; le coût affiché est une estimation. La session est libérée à la fin.

## 0. Commit figé, budget

In [ ]:
# Commit FIGE du code (depot public). Tant qu'il n'est pas renseigne, rien ne tourne : session liberee.
import re as _re
COMMIT_FIGE = "b3ac84a93e34c4bef6a26a07cc95214973fce083"
if not _re.fullmatch(r"[0-9a-f]{40}", COMMIT_FIGE):
    from google.colab import runtime
    print("commit du code non renseigne (scripts/make_national_gfs_v5_nb.py, COMMIT) : rien n'est lance, "
          "session liberee")
    runtime.unassign()
    raise SystemExit("commit non fige")
print("commit fige :", COMMIT_FIGE)

In [ ]:
# BUDGET (regle v5.1) : plafond 90 unites Colab pour toute la regle (A + B1 + B2). Renseigner les unites DEJA
# consommees par la regle (0 avant la partie A) en lisant le solde Colab. Cout de cette partie : ESTIMATION.
UNITES_DEJA_UTILISEES = None
_mat, _taux, _hmin, _hmax = "Colab CPU", 0.1, 3.0, 8.0
_bas, _haut = _taux * _hmin, _taux * _hmax
print(f"cout ESTIME de la partie A (Colab CPU, ~0.1 unites/h, estimation) : {_hmin:g}-{_hmax:g} h, "
      f"soit ~{_bas:.0f}-{_haut:.0f} unites ; plafond de la regle : 90 unites (A + B1 + B2)")
if UNITES_DEJA_UTILISEES is None or float(UNITES_DEJA_UTILISEES) + _haut > 90:
    from google.colab import runtime
    print("budget : UNITES_DEJA_UTILISEES non renseigne, ou plafond risque d'etre depasse : rien n'est lance, "
          "demander a Ines (regle v5.1, « budget ») ; session liberee")
    runtime.unassign()
    raise SystemExit("budget non verifie")
print(f"budget : {float(UNITES_DEJA_UTILISEES):g} deja utilisees + {_haut:.0f} au plus <= 90")

## 1. Drive, code au commit figé, données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "b3ac84a93e34c4bef6a26a07cc95214973fce083"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"), ("regle_national_gfs_v4.json", "4ca1f2a0c34856d6840d620cd9457534c37cb316a59080400e2853e2f0d7d721"), ("regle_national_gfs_v5_1.json", "570713841039a5fda8e4d8727a047a4c6a3171a9c212b5402cdb2b7356be9cdd")):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regles v4, gfs_v4 et gfs_v5_1 verifiees")

GV5 = NAT / "gfs_v5_regle"                 # dossier NEUF de la regle (rien d'autre n'y ecrit)
GV5.mkdir(exist_ok=True)
ECH = (0, 24, 48)
F = {e: GV5 / f"forcage_gfs_f{e}.csv" for e in ECH}                 # entrainement 2023-2025 + validation 2021
TN = {e: GV5 / f"gfs_f{e}_tn" for e in ECH}                         # echeances telechargees (boite nationale)
F26 = {e: GV5 / "jugement_2026" / f"forcage_gfs_2026_f{e}.csv" for e in ECH}
TN26 = {e: GV5 / f"gfs_2026_f{e}_tn" for e in ECH}
# archives deja telechargees, LUES SEULEMENT (rien n'y est ecrit) : OS7 du Grand Tunis, OS7 national s'il existe
LECTURE = {e: [d for d in (GT / "os7" / f"gfs_f{e}_instants", NAT / "os7" / f"gfs_f{e}_tn") if d.exists()] for e in ECH}
VAL = GV5 / "validation_2021"              # etapes 1 + 3
RES = GV5 / "resultat"                     # etape 4 (controle, puis jugement 2026)
METAR26 = GV5 / "metar_2026"               # METAR 2026 (archive IEM), telecharges seulement a l'etape 4
RCP = NAT / "runs" / "national_1km_v4_Cp_s44"          # C' s44 tel que livre (lu seulement)
R5 = NAT / "runs" / "national_1km_v5_gfs"              # C1 : run NEUF
SHA_V5_1 = "570713841039a5fda8e4d8727a047a4c6a3171a9c212b5402cdb2b7356be9cdd"
# echantillon 2026 de la regle v5.1 : jours de l'annee impairs, heures de cycle (analyse toujours f000)
PERIODE_2026 = "--periode 2026-01-01T00:00,2026-10-31T18:00 --jours-impairs --heures 0,6,12,18"
print("accord :", "Ines, 22/11/2026 (soir) : regle_national_gfs_v5_1 approuvee et hachee (570713841039...)")

## 2. GFS (archive NOAA sur AWS), boîte nationale : entraînement 2023-2025 + validation 2021, puis l'échantillon 2026 ; reprise possible ; fin de la session

In [ ]:
# PARTIE A (Colab CPU) : GFS f0 / f24 / f48 aux instants d'entrainement et de validation, puis a l'echantillon 2026
!pip install -q herbie-data cfgrib eccodes 2>&1 | tail -1
from google.colab import runtime
import traceback
def tolerer_code4(nom, cmd):
    # code 4 = plus de 2 % d'echeances absentes de l'archive apres 3 passes : le controle de la regle est fait par les
    # scripts de jugement sur les instants juges (validation 2021, 2026) ; ici on garde ce qui a ete telecharge
    try:
        lancer(nom, cmd)
    except RuntimeError as exc:
        if "(code 4)" not in str(exc):
            raise
        print(exc)
try:
    CACHE = Path("/content/national/cache")          # l'index du cache v3 seulement (pas les ~50 Go)
    CACHE.mkdir(parents=True, exist_ok=True)
    import pandas as pd
    parts = []
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            with zf.open(f"index_partie_{k}.csv") as fi:
                parts.append(pd.read_csv(fi))
    pd.concat(parts).drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
    for e in ECH:                                     # 1. instants d'entrainement et de validation (C1, etapes 1 + 3)
        if F[e].exists():
            print(f"f{e} entrainement + validation deja ecrit : {F[e]}")
            continue
        lecture = " ".join(f"--lecture {d}" for d in LECTURE[e])
        tolerer_code4(f"gfs_v5_f{e}", f"python -u -m pcfno3d.gfs --config national_1km_colab --national --echeance {e} "
                                     f"--splits train,val --sortie {F[e]} --reprise {TN[e]} {lecture} --travailleurs 8")
    for e in ECH:                                     # 2. echantillon 2026 (etape 4 ; GFS seulement, aucune mesure lue)
        if F26[e].exists():
            print(f"f{e} 2026 deja ecrit : {F26[e]}")
            continue
        tolerer_code4(f"gfs_v5_2026_f{e}", f"python -u -m pcfno3d.gfs --config national_1km_colab --national "
                                          f"--echeance {e} {PERIODE_2026} --sortie {F26[e]} --reprise {TN26[e]} "
                                          f"--travailleurs 8")
    for nom, d in (("entrainement + validation", F), ("2026", F26)):
        for e in ECH:
            err = d[e].with_suffix(".erreurs.json")
            print(f"{nom} f{e} :", d[e].exists(), "| echeances absentes :",
                  len(json.loads(err.read_text())) if err.exists() else 0)
    print("partie A terminee : lancer le carnet B1 (A100)")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()